<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula5-exercicio1-pesos-que-ainda-treinam.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 5, exercício 1: Pesos que ainda treinam

A base do MobileNetV2 ficou congelada. Quantos parâmetros de `modelo` ainda podem mudar? É a cabeça nova, a que separa normal de pneumonia.

O exercício é da aula [5. Pneumonia com Keras](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/05-pneumonia-keras.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara o modelo, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q tensorflow

## Preparação

A preparação baixa radiografias de tórax já rotuladas como normal ou pneumonia, congela o MobileNetV2 que veio do ImageNet e treina só a cabeça por duas épocas.

In [ ]:
import random
import shutil
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf

URL_RAIOX = (
    "https://huggingface.co/datasets/keremberke/chest-xray-classification"
    "/resolve/main/data/valid.zip"
)
arquivo = Path("valid.zip")
if not arquivo.exists():
    urllib.request.urlretrieve(URL_RAIOX, arquivo)
if not Path("raiox/NORMAL").exists():
    zipfile.ZipFile(arquivo).extractall("raiox")

rng = random.Random(7)

def partir(classe, n_treino, n_prova):
    fotos = sorted((Path("raiox") / classe).glob("*.jpg"))
    rng.shuffle(fotos)
    return fotos[:n_treino], fotos[n_treino:n_treino + n_prova]

for split in ("treino", "prova"):
    shutil.rmtree(split, ignore_errors=True)
for classe in ("NORMAL", "PNEUMONIA"):
    treino_fotos, prova_fotos = partir(classe, 24, 8)
    for split, fotos in (("treino", treino_fotos), ("prova", prova_fotos)):
        destino = Path(split) / classe
        destino.mkdir(parents=True)
        for foto in fotos:
            shutil.copy(foto, destino / foto.name)

tamanho = (128, 128)
treino = tf.keras.utils.image_dataset_from_directory(
    "treino", image_size=tamanho, batch_size=8, shuffle=False,
)
prova = tf.keras.utils.image_dataset_from_directory(
    "prova", image_size=tamanho, batch_size=8, shuffle=False,
)
nomes = treino.class_names

for imagens, rotulos in treino.take(1):
    plt.figure(figsize=(8, 3))
    for i in range(4):
        plt.subplot(1, 4, i + 1)
        plt.imshow(imagens[i].numpy().astype("uint8"))
        plt.title(nomes[int(rotulos[i])])
        plt.axis("off")
    plt.show()
    break

preparar = tf.keras.applications.mobilenet_v2.preprocess_input
treino_modelo = treino.map(lambda imagem, rotulo: (preparar(imagem), rotulo))
prova_modelo = prova.map(lambda imagem, rotulo: (preparar(imagem), rotulo))

base = tf.keras.applications.MobileNetV2(
    input_shape=tamanho + (3,), include_top=False, weights="imagenet",
)
base.trainable = False
entradas = tf.keras.Input(shape=tamanho + (3,))
x = base(entradas, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
saidas = tf.keras.layers.Dense(len(nomes), activation="softmax")(x)
modelo = tf.keras.Model(entradas, saidas)
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
modelo.fit(treino_modelo, epochs=2, verbose=0)
perda, acerto = modelo.evaluate(prova_modelo, verbose=0)
treinaveis = sum(int(tf.reduce_prod(tf.shape(p)).numpy()) for p in modelo.trainable_weights)
print(f"classes: {nomes}")
print(f"base pode mudar? {base.trainable}")
print(f"pesos que ainda treinam: {treinaveis}")
print(f"acerto na prova: {acerto:.0%}")

In [ ]:
# @title Correção automática (rode, não precisa ler)
import tensorflow as tf

VERIFICAR = {
    1: lambda r: int(r) == sum(int(tf.reduce_prod(tf.shape(p)).numpy()) for p in modelo.trainable_weights),
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

A base do MobileNetV2 ficou congelada. Quantos parâmetros de `modelo` ainda podem mudar? É a cabeça nova, a que separa normal de pneumonia.

*Dica:* modelo.trainable_weights lista só o que o Adam pode mexer. Some o tamanho de cada um.

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(1, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = sum(int(tf.reduce_prod(tf.shape(p)).numpy()) for p in modelo.trainable_weights)

conferir(1, resposta)

## O que fica

Esse foi o último dos 25. O acerto numa radiografia de treino ainda não é um laudo. A prova continua sendo um conjunto que o ajuste não viu.